In [1]:
# Fix Keras compatibility issue and install required packages
import subprocess
import sys

def install_package(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"Successfully installed {package}")
    except subprocess.CalledProcessError as e:
        print(f"Error installing {package}: {e}")

# Install required packages
print("Installing required packages...")
install_package("tf-keras")
install_package("transformers")
install_package("torch")
install_package("torchvision")
install_package("pillow")
install_package("pandas")

print("Package installation complete!")


Installing required packages...
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Package installation complete!
P

In [2]:
# AI Image Detector v2.5 (Ensemble Model)
# This version uses two different, reliable models for a more robust prediction.

import pandas as pd
from PIL import Image
import os
from transformers import pipeline, Pipeline
import torch

# --- Configuration ---
# Using more reliable models that work better with current transformers
MODEL_1 = "Organika/sdxl-detector"
MODEL_2 = "umm-maybe/AI-image-detector"

# --- Model Loading with Robust Error Handling ---
detector1: Pipeline = None
detector2: Pipeline = None
models_loaded = {"model1": False, "model2": False}

def load_models():
    """Load models with individual error handling"""
    global detector1, detector2, models_loaded

    # Try loading Model 1
    try:
        print("Loading Model 1 (Organika/sdxl-detector)...")
        detector1 = pipeline("image-classification", model=MODEL_1, trust_remote_code=True)
        models_loaded["model1"] = True
        print("Model 1 loaded successfully!")
    except Exception as e:
        print(f"Error loading Model 1: {e}")
        detector1 = None

    # Try loading Model 2 with fallback
    try:
        print(f"Loading Model 2 ({MODEL_2})...")
        detector2 = pipeline("image-classification", model=MODEL_2, trust_remote_code=True)
        models_loaded["model2"] = True
        print("Model 2 loaded successfully!")
    except Exception as e:
        print(f"Error loading Model 2: {e}")
        detector2 = None

        # Try a simple fallback model
        try:
            print("Trying fallback model...")
            detector2 = pipeline("image-classification", model="microsoft/resnet-50")
            models_loaded["model2"] = True
            print("Fallback model loaded successfully!")
        except Exception as e2:
            print(f"Fallback model also failed: {e2}")

    # Check if at least one model loaded
    if not any(models_loaded.values()):
        print("CRITICAL: No models loaded. The detector will not function properly.")
        return False
    elif not all(models_loaded.values()):
        print("WARNING: Only one model loaded. Ensemble predictions unavailable.")

    return True

# Load models at startup
print("=" * 60)
print("AI IMAGE DETECTOR - MODEL INITIALIZATION")
print("=" * 60)
load_models()
print("=" * 60)
print()

def normalize_prediction(predictions, model_name):
    """
    Extract AI-generated probability from model predictions.
    Returns: (label, ai_confidence_score)
    """
    # Create a dictionary of label->score for easy lookup
    pred_dict = {pred["label"].lower(): pred["score"] for pred in predictions}

    if "organika" in model_name.lower():
        # Organika model uses different labels
        ai_score = pred_dict.get('ai', pred_dict.get('artificial', 0))
    elif "umm-maybe" in model_name.lower():
        # umm-maybe uses 'artificial' and 'human'
        ai_score = pred_dict.get('artificial', 0)
    elif "resnet" in model_name.lower():
        # ResNet is a general classifier, use top prediction confidence
        ai_score = predictions[0]["score"] if predictions else 0.5
    else:
        # General fallback
        ai_score = pred_dict.get('ai', pred_dict.get('artificial', predictions[0]["score"] if predictions else 0.5))

    # Determine label based on higher score
    label = "AI-Generated" if ai_score > 0.5 else "Human-Generated"
    # Return the AI confidence (probability of being AI-generated)
    return label, ai_score

def validate_image(image):
    """Basic image validation"""
    if image is None:
        return False, "No image provided"

    try:
        # Check if image can be processed
        if not isinstance(image, Image.Image):
            return False, "Invalid image format"

        # Check image size (not too small)
        if image.size[0] < 50 or image.size[1] < 50:
            return False, "Image too small (minimum 50x50 pixels)"

        return True, "Valid"
    except Exception as e:
        return False, f"Image validation error: {str(e)}"

def detect_ai_image(file_path):
    """
    Accepts a file path to an image and returns AI generation probability.
    
    Args:
        file_path: Path to the image file
    
    Returns:
        dict: Contains 'ai_percentage', 'verdict', 'confidence', and 'details'
    """
    result = {
        'ai_percentage': 0.0,
        'verdict': 'Error',
        'confidence': 'N/A',
        'details': {}
    }
    
    # Check if file exists
    if not os.path.exists(file_path):
        result['details']['error'] = f"File not found: {file_path}"
        return result
    
    # Load image
    try:
        image = Image.open(file_path)
    except Exception as e:
        result['details']['error'] = f"Failed to load image: {str(e)}"
        return result
    
    # Validate image
    is_valid, validation_msg = validate_image(image)
    if not is_valid:
        result['details']['error'] = validation_msg
        return result

    # Check if any models are loaded
    if not any(models_loaded.values()):
        result['details']['error'] = "No AI detection models are available"
        return result

    predictions = []

    # Get prediction from Model 1
    if models_loaded["model1"] and detector1:
        try:
            pred1 = detector1(image, top_k=2)
            label1, score1 = normalize_prediction(pred1, MODEL_1)
            predictions.append(("Model 1", label1, score1))
            result['details']['model1_label'] = label1
            result['details']['model1_ai_score'] = f"{score1:.2%}"
        except Exception as e:
            result['details']['model1_error'] = str(e)
            print(f"Model 1 prediction error: {e}")

    # Get prediction from Model 2
    if models_loaded["model2"] and detector2:
        try:
            pred2 = detector2(image, top_k=2)
            label2, score2 = normalize_prediction(pred2, MODEL_2)
            predictions.append(("Model 2", label2, score2))
            result['details']['model2_label'] = label2
            result['details']['model2_ai_score'] = f"{score2:.2%}"
        except Exception as e:
            result['details']['model2_error'] = str(e)
            print(f"Model 2 prediction error: {e}")

    # --- Determine Final Verdict ---
    if len(predictions) == 2:
        # Both models available - ensemble prediction
        label1, score1 = predictions[0][1], predictions[0][2]
        label2, score2 = predictions[1][1], predictions[1][2]

        # Average the AI confidence scores
        avg_score = (score1 + score2) / 2
        result['ai_percentage'] = avg_score * 100

        # Decision boundary: 0.3 is the tipping point
        final_verdict = "AI-Generated" if avg_score > 0.3 else "Human-Generated"
        
        if final_verdict == "AI-Generated":
            if avg_score > 0.8:
                confidence = "High"
            elif avg_score > 0.6:
                confidence = "Medium"
            else:
                confidence = "Low"
        else:
            if avg_score < 0.2:
                confidence = "High"
            else:
                confidence = "Medium"

        result['verdict'] = final_verdict
        result['confidence'] = confidence
        result['details']['agreement'] = "Yes" if label1 == label2 else "No"
        result['details']['ensemble_method'] = "Average of 2 models"

    elif len(predictions) == 1:
        # Only one model available
        model_name, label, score = predictions[0][0], predictions[0][1], predictions[0][2]
        result['ai_percentage'] = score * 100
        result['verdict'] = label
        
        if score > 0.7 or score < 0.3:
            confidence = "Medium"
        else:
            confidence = "Low"
        
        result['confidence'] = confidence
        result['details']['note'] = "Only one model available"

    else:
        result['details']['error'] = "Both models failed to process the image"

    return result

print("AI Image Detector initialized successfully!")
print("Use detect_ai_image(file_path) to analyze an image")
print()


2025-10-11 23:48:13.938757: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-11 23:48:14.041522: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-11 23:48:16.888274: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-11 23:48:16.888274: I tensorflow/core/util/port.cc:153] oneD

AI IMAGE DETECTOR - MODEL INITIALIZATION
Loading Model 1 (Organika/sdxl-detector)...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0
Device set to use cuda:0


Model 1 loaded successfully!
Loading Model 2 (umm-maybe/AI-image-detector)...


Device set to use cuda:0


Model 2 loaded successfully!

AI Image Detector initialized successfully!
Use detect_ai_image(file_path) to analyze an image



## Analyze Image

Specify the path to your image file below and run the cell to get the AI generation probability.

In [3]:
# ============================================
# SPECIFY YOUR IMAGE FILE PATH HERE
# ============================================
file_path = "/home/alookaladdoo/Pictures/phulara.jpg"  # CHANGE THIS TO YOUR IMAGE PATH

# Example paths (uncomment to use):
# file_path = "/home/alookaladdoo/Pictures/test_image.jpg"
# file_path = "sample_image.png"

# Run detection
if os.path.exists(file_path):
    print("=" * 60)
    print(f"ANALYZING IMAGE: {os.path.basename(file_path)}")
    print("=" * 60)
    
    result = detect_ai_image(file_path)
    
    # Display results
    print()
    print("-" * 60)
    print("RESULTS:")
    print("-" * 60)
    print(f"AI Generation Probability: {result['ai_percentage']:.2f}%")
    print(f"Verdict: {result['verdict']}")
    print(f"Confidence: {result['confidence']}")
    print()
    
    if result['details']:
        print("Details:")
        for key, value in result['details'].items():
            print(f"  - {key}: {value}")
    
    print("=" * 60)
    
    # Interpretation
    print()
    if result['verdict'] == 'Error':
        print("ERROR: Could not analyze image")
    elif result['ai_percentage'] > 70:
        print("INTERPRETATION: This image is LIKELY AI-GENERATED")
    elif result['ai_percentage'] > 30:
        print("INTERPRETATION: UNCERTAIN - Manual inspection recommended")
    else:
        print("INTERPRETATION: This image is LIKELY HUMAN-CREATED")
    print()
    
else:
    print("=" * 60)
    print(f"ERROR: File not found at '{file_path}'")
    print("Please update the file_path variable with the correct path")
    print("=" * 60)


ANALYZING IMAGE: phulara.jpg

------------------------------------------------------------
RESULTS:
------------------------------------------------------------
AI Generation Probability: 0.78%
Verdict: Human-Generated
Confidence: High

Details:
  - model1_label: Human-Generated
  - model1_ai_score: 0.00%
  - model2_label: Human-Generated
  - model2_ai_score: 1.56%
  - agreement: Yes
  - ensemble_method: Average of 2 models

INTERPRETATION: This image is LIKELY HUMAN-CREATED


------------------------------------------------------------
RESULTS:
------------------------------------------------------------
AI Generation Probability: 0.78%
Verdict: Human-Generated
Confidence: High

Details:
  - model1_label: Human-Generated
  - model1_ai_score: 0.00%
  - model2_label: Human-Generated
  - model2_ai_score: 1.56%
  - agreement: Yes
  - ensemble_method: Average of 2 models

INTERPRETATION: This image is LIKELY HUMAN-CREATED



---

## How to Use This Notebook

### Quick Start:
1. **Run the first two cells** to install packages and load models
2. **In the "Analyze Image" section**, update the `file_path` variable with your image path
3. **Run the cell** to get the AI generation percentage

### Understanding the Results:

**AI Generation Probability:**
- **0-30%**: Image is likely human-created
- **30-70%**: Uncertain - requires manual inspection
- **70-100%**: Image is likely AI-generated

**Verdict Options:**
- `AI-Generated`: Model predicts the image was created by AI
- `Human-Generated`: Model predicts the image was created by a human
- `Error`: Analysis failed

**Confidence Levels:**
- `High`: Strong prediction (more reliable)
- `Medium`: Moderate prediction (some uncertainty)
- `Low`: Weak prediction (high uncertainty)

### Technical Details:

**Models Used:**
1. **Model 1**: Organika/sdxl-detector (specialized for SDXL AI images)
2. **Model 2**: umm-maybe/AI-image-detector (general AI detection)

**Ensemble Method:**
- When both models are available, their predictions are averaged
- If models agree, confidence is higher
- If models disagree, the result is flagged for manual review

**Supported Formats:**
- JPEG, PNG, BMP, GIF, TIFF
- Minimum image size: 50x50 pixels

---